# GAMMA DNA Tutorial 1 — Prediabetes Classification

Replicating the FACET *Classification with FACET* tutorial using **GAMMA_DNA**.

| | |
|---|---|
| **Dataset** | NHANES Prediabetes Study (`pre_diab_nhanes.csv`) |
| **Target** | `Pre_diab` (1 = prediabetes, 0 = normal) |
| **Flow** | EDA → Feature Engineering → Training → Synergy / Redundancy → Simulation |


In [1]:
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DAPS_BRIX = "C:/Users/Dai Dennis/OneDrive - The Boston Consulting Group, Inc/Desktop/DAPS_Brix"
if DAPS_BRIX not in sys.path:
    sys.path.insert(0, DAPS_BRIX)

BASE_URL = (
    "https://raw.githubusercontent.com/BCG-X-Official/facet/"
    "aada69350b085f8a3dae7900b16db6c79b146f60/sphinx/source/tutorial/"
)

df = pd.read_csv(BASE_URL + "pre_diab_nhanes.csv")
print(f"Shape: {df.shape}")
df.head()


Shape: (4356, 38)


,Age,Gender,Waist_Circumference,Weight,Standing_Height,BMI,Average_SBP,Average_DBP,HDL_Cholesterol,Total_Cholesterol,...,Triglycerides,Uric_acid,Osmolality,Sodium,Potassium,Gamma_glutamyl_transferase,Calcium,Alanine_aminotransferase,Aspartate_aminotransferase,Pre_diab
0,73.0,2.0,NaN,52.0,162.4,19.7,137.333333,86.666667,85.0,201.0,...,88.0,4.2,290.0,142.0,4.1,31.0,10.0,28.0,36.0,1
1,56.0,1.0,123.1,105.0,158.7,41.7,157.333333,82.000000,38.0,226.0,...,327.0,9.1,287.0,143.0,3.3,22.0,9.3,16.0,24.0,0
2,61.0,2.0,110.8,93.4,161.8,35.7,122.666667,80.666667,58.0,168.0,...,68.0,5.1,281.0,140.0,3.9,17.0,9.9,21.0,20.0,1
3,56.0,2.0,85.5,61.8,152.8,26.5,122.000000,72.666667,59.0,278.0,...,262.0,3.5,277.0,139.0,4.0,21.0,9.5,24.0,23.0,0
4,65.0,1.0,93.7,65.3,172.4,22.0,141.333333,77.333333,79.0,173.0,...,39.0,6.3,281.0,140.0,4.8,24.0,9.5,20.0,29.0,0


## Exploratory Data Analysis


In [2]:
print("Target distribution:")
print(df["Pre_diab"].value_counts(), "\n")

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"Columns with missing values ({len(missing)} of {df.shape[1]}):")
print(missing)


Target distribution:
Pre_diab
0    2847
1    1509
Name: count, dtype: int64 

Columns with missing values (27 of 38):
Tried_weight_loss_past_year    499
General_health                 340
Waist_Circumference            198
Average_DBP                    134
Average_SBP                    127
Calcium                         67
Triglycerides                   50
Aspartate_aminotransferase      49
Alanine_aminotransferase        49
Gamma_glutamyl_transferase      48
Potassium                       48
Uric_acid                       48
Sodium                          47
Osmolality                      47
Feel_at_risk_diab               45
BMI                             42
Standing_Height                 38
Total_Cholesterol               38
HDL_Cholesterol                 38
Weight                          35
Sleep_disorder                  10
Sleep_hours                      4
Moderate_work_activity           2
Told_overweight                  2
Trouble_sleeping                 2
High_BP

In [3]:
df.describe().round(2)


,Age,Gender,Waist_Circumference,Weight,Standing_Height,BMI,Average_SBP,Average_DBP,HDL_Cholesterol,Total_Cholesterol,...,Triglycerides,Uric_acid,Osmolality,Sodium,Potassium,Gamma_glutamyl_transferase,Calcium,Alanine_aminotransferase,Aspartate_aminotransferase,Pre_diab
count,4356.00,4356.00,4158.00,4321.00,4318.00,4314.00,4229.00,4222.00,4318.00,4318.00,...,4306.00,4308.00,4309.00,4309.00,4308.00,4308.00,4289.00,4307.00,4307.00,4356.00
mean,47.04,1.52,97.17,79.94,167.37,28.45,121.89,70.28,53.72,190.36,...,143.41,5.39,279.06,139.88,4.00,27.46,9.45,24.83,25.30,0.35
std,17.19,0.50,15.80,21.14,10.16,6.79,17.27,11.04,16.24,39.63,...,105.11,1.38,4.83,2.21,0.35,46.81,0.36,19.53,19.61,0.48
min,20.00,1.00,55.50,32.30,136.30,14.10,64.67,22.67,10.00,69.00,...,20.00,0.70,237.00,119.00,2.80,5.00,7.60,6.00,9.00,0.00
25%,33.00,1.00,85.90,65.10,160.00,23.70,110.00,63.33,42.00,163.00,...,76.00,4.40,276.00,139.00,3.80,13.00,9.20,16.00,19.00,0.00
50%,45.00,2.00,95.50,76.90,167.10,27.30,118.67,70.67,51.00,188.00,...,113.00,5.30,279.00,140.00,4.00,18.00,9.40,20.00,22.00,0.00
75%,61.00,2.00,106.50,91.10,174.50,31.80,131.33,77.33,63.00,214.00,...,176.00,6.30,282.00,141.00,4.20,28.00,9.70,27.00,27.00,1.00
max,80.00,2.00,177.90,222.60,199.40,82.90,228.00,128.00,173.00,525.00,...,1213.00,13.30,323.00,154.00,5.80,1510.00,14.80,536.00,882.00,1.00


## Feature Engineering

Derived features from the FACET paper:
- **`SBP_to_DBP`** — pulse pressure ratio (systolic / diastolic)
- **`Waist_to_hgt`** — waist-to-height ratio (stronger predictor than raw BMI)


In [4]:
df["SBP_to_DBP"]  = df["Average_SBP"] / df["Average_DBP"]
df["Waist_to_hgt"] = df["Waist_Circumference"] / df["Standing_Height"]

print("Engineered features added.")
df[["SBP_to_DBP", "Waist_to_hgt"]].describe().round(4)


Engineered features added.


,SBP_to_DBP,Waist_to_hgt
count,4222.0000,4152.0000
mean,1.7657,0.5811
std,0.3315,0.0952
min,1.2727,0.3644
25%,1.5501,0.5141
50%,1.6844,0.5711
75%,1.8803,0.6357
max,5.4706,1.0640


## Model Training

GAMMA_DNA auto-detects the task (binary classification), handles missing-value imputation,
and wraps the model in a preprocessing pipeline.


In [5]:
from gamma.pipeline import GAMMA_DNA

g = GAMMA_DNA(df, target="Pre_diab")
result = g.train(model_type="gradient_boosting_classifier")
result.summary()



  ModelResult: gradient_boosting_classifier  [binary_classification]
  Metric                                Train         Test
  --------------------------------------------------------
  accuracy                             0.8014       0.6881
  f1                                   0.7932       0.6665
  precision                            0.7995       0.6702
  recall                               0.8014       0.6881
  roc_auc                              0.8651       0.7280


In [6]:
result.plot()
print(f"\nTest AUC  : {result.roc_auc:.4f}")
print(f"Accuracy  : {result.accuracy:.4f}")
print(f"F1 Score  : {result.f1:.4f}")
print(f"Precision : {result.precision:.4f}")
print(f"Recall    : {result.recall:.4f}")



Test AUC  : 0.7280
Accuracy  : 0.6881
F1 Score  : 0.6665
Precision : 0.6702
Recall    : 0.6881


## Multi-Model Comparison

Compare Logistic Regression, Random Forest, and Gradient Boosting side-by-side
with test AUC and 5-fold cross-validation.


In [7]:
comparison = g.compare_models()
comparison


,model_type,train_score,test_score,cv_mean,cv_std,metric
0,random_forest_classifier,1.0,0.7384,0.7227,0.0082,roc_auc


## SHAP Feature Importance

TreeSHAP values show how each feature contributes to the model's predictions.


In [8]:
report = g.explain(compute_shap=True, compute_permutation=False)
report.plot_overview(max_display=15)


  Computing SHAP values on 39 features… (may take 10-60s)
  Done.


In [9]:
imp = report.to_frame()
imp.sort_values("score", ascending=False).head(15)


,model_imp,perm_imp_mean,shap_mean_abs,rank,score
feature,,,,,
Age,0.283673,NaN,0.553913,1,0.553913
RBC_count,0.063082,NaN,0.204317,2,0.204317
Waist_to_hgt,0.071843,NaN,0.162344,3,0.162344
Gamma_glutamyl_transferase,0.038111,NaN,0.125661,4,0.125661
Uric_acid,0.054937,NaN,0.115511,5,0.115511
HDL_Cholesterol,0.036122,NaN,0.104198,6,0.104198
Triglycerides,0.033083,NaN,0.101178,7,0.101178
BMI,0.042862,NaN,0.093863,8,0.093863
Waist_Circumference,0.055140,NaN,0.084500,9,0.084500


## Feature Synergy Analysis

Synergy measures how much two features *together* explain more than each alone.
A high synergy score means the features are better used jointly than separately.


In [10]:
report.plot_synergy(kind="both", sample_size=200)


## Feature Redundancy Analysis

Redundancy detects features that carry the same information.
We expect **BMI** and **Waist_Circumference** to be redundant with the engineered **Waist_to_hgt**.


In [11]:
report.plot_redundancy(kind="both")


## Drop Redundant Features & Retrain

`g.clean()` removes columns from the session's DataFrame and feature list in-place.


In [12]:
g.clean(drop_cols=["BMI", "Waist_Circumference"])

result2 = g.train(model_type="gradient_boosting_classifier")
print(f"AUC (full model)      : {result.roc_auc:.4f}")
print(f"AUC (reduced model)   : {result2.roc_auc:.4f}")

report2 = g.explain(compute_shap=True, compute_permutation=False)
report2.plot_redundancy(kind="both")


AUC (full model)      : 0.7280
AUC (reduced model)   : 0.7280
  Computing SHAP values on 37 features… (may take 10-60s)
  Done.


## Partial Dependence Plots

PDPs show the marginal effect of one feature on the predicted probability,
averaging over all other features.


In [13]:
report2.plot_pdp("Age", grid_points=30)


In [14]:
report2.plot_pdp("Waist_to_hgt", grid_points=30)


## Simulation — Feature Impact with Confidence Bands

`plot_simulate()` sweeps a feature across a grid, predicts on all test rows at each value,
and shows the **2.5th / mean / 97.5th percentile** prediction bands.


In [15]:
waist_grid = np.linspace(0.40, 0.82, 25)
report2.plot_simulate("Waist_to_hgt", waist_grid)


In [16]:
age_grid = np.arange(20, 80, 5)
report2.plot_simulate("Age", age_grid)


## Top SHAP Feature Interactions


In [17]:
report2.plot_top_interactions(top_k=10, sample_size=200)


In [18]:
top_int = report2.top_interactions(top_k=10, sample_size=200, n_bootstrap=50)
top_int


,feature_a,feature_b,mean_abs_interaction,ci_lower,ci_upper
0,RBC_count,Hematocrit,0.029913,0.027679,0.032991
1,Age,Waist_to_hgt,0.029245,0.025351,0.032301
2,Age,RBC_count,0.024467,0.022384,0.026594
3,Age,Uric_acid,0.024363,0.021311,0.027368
4,Gamma_glutamyl_transferase,Waist_to_hgt,0.022263,0.020828,0.023478
5,Age,Average_DBP,0.017693,0.016526,0.019796
6,Potassium,Waist_to_hgt,0.013664,0.011860,0.015518
7,RBC_count,Waist_to_hgt,0.013628,0.012708,0.014293
8,Average_SBP,Waist_to_hgt,0.013054,0.011423,0.014616
9,Age,Gamma_glutamyl_transferase,0.013030,0.011494,0.014198


## Facet View — Redundancy & Synergy Side-by-Side


In [19]:
report2.plot_facet(sample_size=200)


## Redundancy Linkage Chart


In [20]:
report2.plot_redundancy(kind="linkage")
